# OpenAI + LangSmith Simple Agent
This notebook builds a minimal LangChain agent that sends a question to OpenAI and optionally records the request in LangSmith.

## Workflow
1. Install the required packages.
2. Enter API keys securely at runtime (keys are never saved in the notebook).
3. Configure a `ChatOpenAI` model and a reusable prompt.
4. Invoke the traceable helper function with a question and print the response.

## 1. Install dependencies
Run the next cell once to install LangChain, the OpenAI integration, and LangSmith support in the active notebook environment. Restart the kernel if the package installation requests it.

In [32]:
%pip install -q langchain langchain-openai langchain-community langsmith

Note: you may need to restart the kernel to use updated packages.


In [33]:
import os
from getpass import getpass
from langsmith import traceable
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage

## 2. Configure credentials and tracing
The next cell prompts for your OpenAI API key (`sk-...`) without displaying it or writing it into this notebook. It also asks for a LangSmith key to enable tracing.

> **Security:** Do not paste keys into code or commit them to Git. If you do not use LangSmith, remove the tracing lines and the `LANGCHAIN_API_KEY` prompt.

In [34]:
# ---------- 1) Prompt for secrets (one-time) ----------
os.environ.pop("OPENAI_API_KEY", None)
os.environ.pop("LANGCHAIN_API_KEY", None)
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key (sk-...): ")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass("Enter your LANGCHAIN_API_KEY: ")
os.environ.setdefault("LANGCHAIN_PROJECT", "OpenAI-LLM-Demo")
# Sanity check (no secrets printed)
print(
    "Tracing on?:", os.getenv("LANGCHAIN_TRACING_V2") == "true",
    "| Project:", os.getenv("LANGCHAIN_PROJECT"),
    "| Has OpenAI key?:", bool(os.getenv("OPENAI_API_KEY")),
    "| Has LangSmith key?:", bool(os.getenv("LANGCHAIN_API_KEY")),
)

Tracing on?: True | Project: Azure-LLM-Demo | Has OpenAI key?: True | Has LangSmith key?: True


In [35]:
# ---------- 2) OpenAI Chat Model ----------
llm = ChatOpenAI(
    model="gpt-5-mini",
    api_key=os.environ["OPENAI_API_KEY"],
)

In [36]:
# ---------- 3) Prompt ----------
prompt = PromptTemplate.from_template("Answer clearly and concisely:\n{question}")

## 3. Build the agent response
`ChatOpenAI` sends requests to the selected OpenAI model. The prompt template inserts the supplied question, and `simple_agent_response` invokes the model with a `HumanMessage`.

The `@traceable` decorator records the function run in LangSmith when tracing is configured, which makes it easier to inspect prompts, responses, and latency.

In [37]:
# ---------- 4) Traceable function ----------
@traceable(name="SimpleAgentTrace")
def simple_agent_response(question: str) -> str:
 formatted_prompt = prompt.format(question=question)
 response = llm.invoke(
  [HumanMessage(content=formatted_prompt)]
 )
 return response.content

## 4. Run an example
Run the final cell to submit the example question and print the model response. Change the value of `q` to ask a different question.

**Note:** You need an active OpenAI account with access to the configured model. If `gpt-5-mini` is unavailable to your account, replace it in the model-configuration cell with a model you can access.

In [38]:
# ---------- 5) Run ----------
if __name__ == "__main__":
    q = "What are the key benefits of observability in AI agents?"
    print("Question:", q)
    answer = simple_agent_response(q)
    print("\nAnswer:\n", answer)

Question: What are the key benefits of observability in AI agents?

Answer:
 - Faster debugging and root-cause analysis: detailed traces, logs, and traces let engineers find where and why an agent failed or behaved unexpectedly.  
- Shorter development cycles: observable signals speed iteration on prompts, chain-of-thoughts, tools, and system design.  
- Improved reliability and uptime: real-time metrics and alerts detect degradations so teams can respond before major failures.  
- Safety and alignment: behavior traces and provenance make it easier to detect unsafe or misaligned actions and enforce guardrails.  
- Transparency and explainability: recorded decisions, inputs, and intermediate steps support understanding and communicating why an agent acted a certain way.  
- Compliance and auditability: immutable logs and traces provide evidence for regulatory, legal, or internal audit requirements.  
- Performance optimization: telemetry identifies bottlenecks (latency, throughput, reso